# Data Cleaning and Transformation

This notebook performs data cleaning, validation and transformation
for all mutual fund datasets before loading into SQLite and analytics workflows.


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

Load Datasets

In [4]:
raw_dir = Path("../data/raw")

fund_master = pd.read_csv(raw_dir/"01_fund_master.csv")
nav_history = pd.read_csv(raw_dir/"02_nav_history.csv")
aum = pd.read_csv(raw_dir/"03_aum_by_fund_house.csv")
sip = pd.read_csv(raw_dir/"04_monthly_sip_inflows.csv")
category = pd.read_csv(raw_dir/"05_category_inflows.csv")
folio = pd.read_csv(raw_dir/"06_industry_folio_count.csv")
performance = pd.read_csv(raw_dir/"07_scheme_performance.csv")
transactions = pd.read_csv(raw_dir/"08_investor_transactions.csv")
holdings = pd.read_csv(raw_dir/"09_portfolio_holdings.csv")
benchmark = pd.read_csv(raw_dir/"10_benchmark_indices.csv")

Fund Master Cleaning

## Fund Master Cleaning

- Remove duplicate schemes
- Convert launch date
- Standardize text fields

In [5]:
fund_master = fund_master.drop_duplicates(
    subset="amfi_code"
)

fund_master["launch_date"] = pd.to_datetime(
    fund_master["launch_date"],
    errors="coerce"
)

fund_master.head()

,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


NAV Cleaning

## NAV History Cleaning

- Convert date
- Convert NAV to numeric
- Remove invalid NAV values
- Forward fill missing NAVs

In [6]:
nav_history["date"] = pd.to_datetime(
    nav_history["date"]
)

nav_history["nav"] = pd.to_numeric(
    nav_history["nav"],
    errors="coerce"
)

nav_history = nav_history[
    nav_history["nav"] > 0
]

nav_history = nav_history.sort_values(
    ["amfi_code","date"]
)

nav_history["nav"] = (
    nav_history.groupby("amfi_code")
    ["nav"]
    .ffill()
)

AUM Cleaning

In [7]:
aum["date"] = pd.to_datetime(
    aum["date"]
)

aum["aum_crore"] = pd.to_numeric(
    aum["aum_crore"],
    errors="coerce"
)

aum = aum[
    aum["aum_crore"] > 0
]

SIP Cleaning

In [8]:
sip["month"] = pd.to_datetime(
    sip["month"]
)

Category Cleaning

In [9]:
category["month"] = pd.to_datetime(
    category["month"]
)

category["category"] = (
    category["category"]
    .str.strip()
)

Performance Cleaning

In [10]:
performance = performance[
    performance["expense_ratio_pct"]
    .between(0.1,2.5)
]

Transactions Cleaning

In [11]:
transactions["transaction_date"] = (
    pd.to_datetime(
        transactions["transaction_date"]
    )
)

transactions = transactions[
    transactions["amount_inr"] > 0
]

Holdings Cleaning

In [12]:
holdings["portfolio_date"] = (
    pd.to_datetime(
        holdings["portfolio_date"]
    )
)

holdings = holdings[
    (holdings["weight_pct"] > 0)
    &
    (holdings["weight_pct"] <= 100)
]

Benchmark Cleaning

In [13]:
benchmark["date"] = pd.to_datetime(
    benchmark["date"]
)

benchmark["close_value"] = pd.to_numeric(
    benchmark["close_value"]
)

Validation Checks

In [14]:
print(
    "Fund Master:",
    fund_master.shape
)

print(
    "NAV:",
    nav_history.shape
)

print(
    "Performance:",
    performance.shape
)

Fund Master: (40, 15)
NAV: (46000, 3)
Performance: (40, 19)


AMFI Validation

In [15]:
fund_codes = set(
    fund_master["amfi_code"]
)

nav_codes = set(
    nav_history["amfi_code"]
)

print(
    "Missing Codes:",
    len(
        nav_codes - fund_codes
    )
)

Missing Codes: 0


Data Quality Summary

## Data Quality Summary

- Duplicate records removed
- Date fields standardized
- Invalid NAV values removed
- Missing values handled
- Expense ratio validated
- Portfolio weights validated
- AMFI code consistency verified

The datasets are now ready for EDA, Performance Analytics and Dashboard Development.